# Módulo de limpeza e transformação para identificação de desertos médicos Sus

**Autora:** Vanessa Batista ([@vandromedae](https://github.com/vandromedae))  
**Repositório:** [github.com/vandromedae/desertos-medicos-sus](https://github.com/vandromedae/desertos-medicos-sus)  
**Licença:** [MIT](https://github.com/vandromedae/desertos-medicos-sus/blob/main/LICENSE)

---

In [ ]:
# ============================================================
# Desertos Médicos SUS - Limpeza e Transformação
# ============================================================

import sys
from pathlib import Path

# Adicionar raiz do projeto ao path
notebook_path = Path().resolve()
project_root = notebook_path.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DATA_EXTERNAL, DATA_PROCESSED,
    COMPETENCIA_DEFAULT, LIMIAR_DENSIDADE_MEDICA,
    BRAZIL_LON_RANGE, BRAZIL_LAT_RANGE,
)
from src.analysis import classificar_densidade_medica
from src.geospatial import parse_location_wkt
from src.preprocessing import (
    padronizar_campos_medicos,
    carregar_censo_sp,
    agregar_medicos_por_municipio,
    cruzar_medicos_populacao,
)

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print(" Ambiente configurado!")
print(f" Raiz do projeto: {project_root}")
print(f" Dados processados: {DATA_PROCESSED}")
print(f" Dados externos: {DATA_EXTERNAL}")

In [ ]:
# ============================================================
# Carregar base de médicos únicos (gerada no notebook 01)
# ============================================================

arquivo_medicos = DATA_PROCESSED / "medicos_sus_unicos.parquet"

if not arquivo_medicos.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {arquivo_medicos}\n"
        "Execute o notebook 01_coleta_e_exploracao.ipynb primeiro."
    )

df_medicos = pd.read_parquet(arquivo_medicos)

print(f" {len(df_medicos):,} médicos únicos carregados")
print(f" Colunas disponíveis: {len(df_medicos.columns)}")
print(f"\n Amostra dos dados:")
display(df_medicos.head())

In [ ]:
# ============================================================
# Diagnóstico de qualidade dos dados
# ============================================================

print("=" * 70)
print("DIAGNÓSTICO DE QUALIDADE")
print("=" * 70)

# Campos críticos para análise
campos_criticos = [
    'cnes', 'municipio', 'uf', 'profissional_cns',
    'profissional_cbo', 'convenio_sus', 'location'
]

print("\n Percentual de nulos em campos críticos:")
for campo in campos_criticos:
    if campo in df_medicos.columns:
        nulos = df_medicos[campo].isna().sum()
        pct = (nulos / len(df_medicos)) * 100
        status = "⚠️" if pct > 5 else "✅"
        print(f"  {status} {campo:30s}: {nulos:>8,} ({pct:.2f}%)")
    else:
        print(f"  {campo:30s}: COLUNA NÃO EXISTE")

# Estatísticas básicas
print(f"\n Estatísticas básicas:")
print(f"  CNES únicos: {df_medicos['cnes'].nunique():,}")
print(f"  Municípios: {df_medicos['municipio'].nunique():,}")
print(f"  UFs: {df_medicos['uf'].nunique():,}")
print(f"  Médicos únicos (CNS): {df_medicos['profissional_cns'].nunique():,}")

# Distribuição de CBO
print(f"\n Top 10 CBOs (especialidades):")
top_cbos = df_medicos['profissional_cbo'].value_counts().head(10)
for cbo, count in top_cbos.items():
    print(f"  {cbo:50s}: {count:>8,}")

In [ ]:
# ============================================================
# Padronização de campos críticos
# ============================================================
# versão original, mantida para conferência:
# df = df_medicos.copy()
# df["cnes"] = df["cnes"].astype(str).str.strip().str.zfill(7)
# df["municipio"] = df["municipio"].astype(str).str.strip().str.upper()
# df["uf"] = df["uf"].astype(str).str.strip().str.upper()
# df["profissional_cns"] = df["profissional_cns"].astype(str).str.strip()
# if "ibge" in df.columns:
#     df["cod_mun_ibge"] = df["ibge"].astype(str).str.strip().str.zfill(6)

df = padronizar_campos_medicos(df_medicos)

print(f" CNES padronizado (ex: {df['cnes'].iloc[0]})")
print(f" Município padronizado (ex: {df['municipio'].iloc[0]})")
print(f" UF padronizado: {df['uf'].unique()}")
print("CNS padronizado")
print(f"Código IBGE do município criado (ex: {df['cod_mun_ibge'].iloc[0]})")
print(f"\nApós padronização:")
print(f"  CNES únicos: {df['cnes'].nunique():,}")
print(f"  Municípios únicos: {df['municipio'].nunique():,}")
print(f"  Códigos IBGE únicos: {df['cod_mun_ibge'].nunique():,}")

In [ ]:
# ============================================================
# Extrair coordenadas da coluna 'location' (formato WKT)
# ============================================================
# Formato esperado: "POINT (longitude latitude)"
# Exemplo: "POINT (-46.6333 -23.5505)"

# versão original, mantida para conferência:
# def extrair_coordenadas(location_str):
#     ...

# Aplicar extração
print(" Extraindo coordenadas da coluna 'location'...")
df[['longitude', 'latitude']] = df['location'].apply(
    lambda x: pd.Series(parse_location_wkt(x))
)

# Converter para numérico
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')

# Validar coordenadas (Brasil: lon entre -74 e -34, lat entre -34 e 5)
mask_lat_valida = df['latitude'].between(BRAZIL_LAT_RANGE[0], BRAZIL_LAT_RANGE[1])
mask_lon_valida = df['longitude'].between(BRAZIL_LON_RANGE[0], BRAZIL_LON_RANGE[1])
df['coordenada_valida'] = mask_lat_valida & mask_lon_valida

# Estatísticas
total = len(df)
com_coords = df['coordenada_valida'].sum()
sem_coords = total - com_coords

print(f"\nCobertura de coordenadas:")
print(f"  Com coordenadas válidas: {com_coords:,} ({com_coords/total*100:.1f}%)")
print(f"  Sem coordenadas ou inválidas: {sem_coords:,} ({sem_coords/total*100:.1f}%)")

# Preview
print(f"\n Amostra de coordenadas extraídas:")
amostra = df[df['coordenada_valida']].sample(5, random_state=42)
display(amostra[['cnes', 'municipio', 'latitude', 'longitude', 'location']].head())

In [ ]:
# ============================================================
# Carregar população do Censo 2022 via Shapefile (IBGE)
# ============================================================

# versão original, mantida para conferência:
# shapefile_path = DATA_EXTERNAL / "SP_setores_CD2022_IBGE" / "SP_setores_CD2022.shp"
# gdf_censo = gpd.read_file(shapefile_path)
# COLUNAS_CENSO = ["CD_SETOR", "CD_MUN", "NM_MUN", "v0001", "AREA_KM2"]
# df_censo["cod_mun_ibge"] = df_censo["CD_MUN"].astype(str).str[:6]

print(f" Carregando shapefile dos setores censitários...")
gdf_censo, df_censo = carregar_censo_sp()
print(f" Shapefile carregado: {len(gdf_censo):,} setores")
print(f" CRS: {gdf_censo.crs}")

print(f"\n Resumo do Censo 2022 (via shapefile):")
print(f"   Setores censitários: {len(df_censo):,}")
print(f"   Municípios: {df_censo['cod_mun_ibge'].nunique():,}")
print(f"   População total: {df_censo['v0001'].sum():,}")

# Diagnóstico rápido de qualidade
nulos_pop = df_censo['v0001'].isna().sum()
if nulos_pop > 0:
    print(f"   {nulos_pop:,} setores sem população ({nulos_pop/len(df_censo)*100:.2f}%)")
else:
    print(f"   Todos os setores têm população registrada")

display(df_censo.head())

In [ ]:
# ============================================================
# Agregar médicos por município
# ============================================================

# versão original, mantida para conferência:
# df_medicos_mun = (
#     df.groupby(["cod_mun_ibge", "municipio", "uf"])
#     .agg(total_medicos=("profissional_cns", "nunique"), ...)
#     .reset_index()
# )

print("=" * 70)
print("AGREGAÇÃO DE MÉDICOS POR MUNICÍPIO")
print("=" * 70)

df_medicos_mun = agregar_medicos_por_municipio(df)

print(f"\n Agregação concluída:")
print(f"   Municípios com médicos: {len(df_medicos_mun):,}")
print(f"   Total de médicos únicos: {df_medicos_mun['total_medicos'].sum():,}")
print(f"   Total de CNES: {df_medicos_mun['total_cnes'].sum():,}")

# Estatísticas
print(f"\n Distribuição de médicos por município:")
print(f"   Média: {df_medicos_mun['total_medicos'].mean():.1f}")
print(f"   Mediana: {df_medicos_mun['total_medicos'].median():.1f}")
print(f"   Máximo: {df_medicos_mun['total_medicos'].max():,}")
print(f"   Mínimo: {df_medicos_mun['total_medicos'].min():,}")

# Top 10 municípios
print(f"\n🏆 Top 10 municípios com mais médicos SUS:")
top10 = df_medicos_mun.nlargest(10, 'total_medicos')[
    ['municipio', 'uf', 'total_medicos', 'total_cnes']
]
display(top10)

In [ ]:
# ============================================================
# Cruzar médicos com população e calcular densidade
# ============================================================

if 'df_censo' not in globals():
    raise RuntimeError("Execute a célula do Censo 2022 (shapefile) primeiro!")

# versão original, mantida para conferência:
# df_pop_mun = (df_censo.groupby(["cod_mun_ibge", "NM_MUN"]).agg(...))
# df_base = pd.merge(df_pop_mun, df_medicos_mun, on="cod_mun_ibge", how="left")
# df_base["medicos_por_1k"] = (df_base["total_medicos"] / pop_segura * 1000).round(2)
# df_base["categoria_densidade"] = df_base["medicos_por_1k"].apply(classificar_densidade_medica)

print("=" * 70)
print("CRUZAMENTO: MÉDICOS + POPULAÇÃO")
print("=" * 70)

df_base = cruzar_medicos_populacao(df_medicos_mun, df_censo)

print(f"\n Merge concluído: {len(df_base):,} municípios")

print(f"\n Densidade médica calculada:")
print(f"   Média estadual: {df_base['medicos_por_1k'].mean():.2f} médicos/1k hab")
print(f"   Mediana: {df_base['medicos_por_1k'].median():.2f}")

# 5. Distribuição por categoria
print(f"\n Distribuição por categoria de densidade:")
distribuicao = df_base.groupby('categoria_densidade').agg(
    num_municipios=('nm_mun', 'count'),
    populacao_total=('populacao', 'sum'),
    medicos_total=('total_medicos', 'sum')
).assign(
    pct_municipios=lambda x: (x['num_municipios'] / x['num_municipios'].sum() * 100).round(1),
    pct_populacao=lambda x: (x['populacao_total'] / x['populacao_total'].sum() * 100).round(1)
)
display(distribuicao)

In [ ]:
# ============================================================
#  Visualizações de validação
# ============================================================

# 1. Corrigir a falta da coluna 'uf' extraindo do código IBGE (primeiros 2 dígitos)
if 'uf' not in df_base.columns:
    df_base['uf'] = df_base['cod_mun_ibge'].astype(str).str[:2]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Desertos Médicos SUS - Validação da Base Municipal', 
             fontsize=14, fontweight='bold')

# 2. Histograma de densidade médica
ax1 = axes[0, 0]
# Tratar possíveis infinitos ou nulos para o histograma
hist_data = df_base['medicos_por_1k'].replace([np.inf, -np.inf], np.nan).dropna()
hist_data.hist(bins=50, ax=ax1, color='steelblue', edgecolor='black')
ax1.axvline(LIMIAR_DENSIDADE_MEDICA['critico'], color='red', 
            linestyle='--', label=f'Crítico ({LIMIAR_DENSIDADE_MEDICA["critico"]})')
ax1.axvline(LIMIAR_DENSIDADE_MEDICA['insuficiente'], color='orange', 
            linestyle='--', label=f'Insuficiente ({LIMIAR_DENSIDADE_MEDICA["insuficiente"]})')
ax1.set_xlabel('Médicos por 1.000 habitantes')
ax1.set_ylabel('Número de municípios')
ax1.set_title('Distribuição de densidade médica')
ax1.legend()

# 3. Boxplot por UF (Top 10 medianas)
ax2 = axes[0, 1]
top_ufs = df_base.groupby('uf')['medicos_por_1k'].median().nlargest(10).index
df_plot_uf = df_base[df_base['uf'].isin(top_ufs)].copy()
df_plot_uf.boxplot(
    column='medicos_por_1k', by='uf', ax=ax2, rot=45
)
ax2.set_title('Densidade médica por UF (Top 10 medianas)')
ax2.set_ylabel('Médicos/1k hab')
ax2.set_xlabel('UF')

# 4. Scatter: População vs Médicos
ax3 = axes[1, 0]
df_plot = df_base[df_base['populacao'] > 0].copy()
ax3.scatter(
    df_plot['populacao'] / 1000,
    df_plot['total_medicos'],
    alpha=0.3, s=20, color='steelblue', edgecolors='none'
)
ax3.set_xlabel('População (milhares)')
ax3.set_ylabel('Total de médicos SUS')
ax3.set_title('Relação População x Médicos (Escala Log)')
ax3.set_xscale('log')
ax3.set_yscale('log')

# 5. Pizza: Categorias de densidade
ax4 = axes[1, 1]
cores_categorias = {
    '1. Crítico (<1,0)': '#d73027',
    '2. Insuficiente (1-2)': '#fc8d59',
    '3. Adequado (2-4)': '#fee08b',
    '4. Bom (4-8)': '#91cf60',
    '5. Excelente (>=8)': '#1a9850',
    'Sem dados': '#cccccc'
}

counts = df_base['categoria_densidade'].value_counts()

# Filtrar apenas categorias que temos cores mapeadas
mask = counts.index.isin(cores_categorias.keys())
counts_plot = counts[mask]
colors_plot = [cores_categorias[cat] for cat in counts_plot.index]

if not counts_plot.empty:
    ax4.pie(counts_plot, labels=counts_plot.index, colors=colors_plot, 
            autopct='%1.1f%%', startangle=90)
ax4.set_title('Distribuição de municípios por categoria')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Identificar municípios em situação de deserto médico
# ============================================================

print("=" * 70)
print("MUNICÍPIOS EM DESERTO MÉDICO (densidade < 1,0 médico/1k hab)")
print("=" * 70)

# Filtrar desertos médicos
df_desertos = df_base[
    (df_base['medicos_por_1k'] < LIMIAR_DENSIDADE_MEDICA['critico']) &
    (df_base['populacao'] > 0)
].copy()

print(f"\n️ Total de municípios em deserto médico: {len(df_desertos):,}")
print(f"   População afetada: {df_desertos['populacao'].sum():,}")
print(f"   % dos municípios: {len(df_desertos)/len(df_base)*100:.1f}%")
print(f"   % da população: {df_desertos['populacao'].sum()/df_base['populacao'].sum()*100:.1f}%")

# Top 20 municípios com maior população em deserto médico
print(f"\n Top 20 municípios mais populosos em deserto médico:")
top_desertos = df_desertos.nlargest(20, 'populacao')[
    ['nm_mun', 'uf', 'populacao', 'total_medicos', 'medicos_por_1k']
]
display(top_desertos)

# Municípios com 0 médicos mas com população
df_zero_medicos = df_base[
    (df_base['total_medicos'] == 0) &
    (df_base['populacao'] > 1000)
].copy()

print(f"\n Municípios com >1000 hab e ZERO médicos SUS: {len(df_zero_medicos):,}")
if len(df_zero_medicos) > 0:
    display(df_zero_medicos.nlargest(10, 'populacao')[
        ['nm_mun', 'uf', 'populacao']
    ])

In [ ]:
# ============================================================
# Salvar base final municipal
# ============================================================

print("=" * 70)
print("SALVANDO BASE FINAL")
print("=" * 70)

# 1. Base municipal completa
output_base_municipal = DATA_PROCESSED / "base_municipal_densidade_medica.parquet"
df_base.to_parquet(output_base_municipal, index=False)
print(f"✅ Base municipal salva: {output_base_municipal.name}")
print(f"   Registros: {len(df_base):,}")

# 2. Base de médicos com coordenadas (para análise geoespacial)
df_medicos_geo = df[df['coordenada_valida']].copy()
output_medicos_geo = DATA_PROCESSED / "medicos_sus_com_coordenadas.parquet"
df_medicos_geo.to_parquet(output_medicos_geo, index=False)
print(f"✅ Médicos com coordenadas salvos: {output_medicos_geo.name}")
print(f"   Registros: {len(df_medicos_geo):,}")

# 3. Relatório em CSV (para fácil visualização)
output_csv = DATA_PROCESSED / "base_municipal_densidade_medica.csv"
df_base.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"✅ CSV exportado: {output_csv.name}")

print(f"\n📊 Resumo final:")
print(f"   Total de municípios: {len(df_base):,}")
print(f"   População total: {df_base['populacao'].sum():,}")
print(f"   Médicos SUS mapeados: {df_base['total_medicos'].sum():,}")
print(f"   Densidade média: {df_base['medicos_por_1k'].mean():.2f} médicos/1k hab")
print(f"   Municípios em deserto médico: {len(df_desertos):,}")